In [1]:
import os
from PIL import Image
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
from collections import Counter
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
from tensorflow.keras.applications import VGG16, ResNet50, MobileNetV2
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.applications.vgg16 import preprocess_input as vgg_pre
from tensorflow.keras.applications.resnet50 import preprocess_input as res_pre
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mob_pre
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import numpy as np

In [2]:
train_dir = "Dataset/train"
val_dir = "Dataset/val"

In [3]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    horizontal_flip=True,
    zoom_range=0.2
)
val_datagen = ImageDataGenerator(rescale=1./255)

In [7]:
train_ds = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    batch_size=64,
    class_mode="binary"      
)
val_ds = val_datagen.flow_from_directory(
    val_dir,
    target_size=(224, 224),
    batch_size=64,
    class_mode="binary"      
)
print("Số lớp:", train_ds.num_classes)


Found 14164 images belonging to 2 classes.
Found 1201 images belonging to 2 classes.
Số lớp: 2


In [8]:
base_model = MobileNetV2(
    weights='imagenet', 
    include_top=False,  
    input_shape=(224, 224, 3)
)


base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(1, activation='sigmoid')  
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [11]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)


c:\Users\vhgam\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
222/222 ━━━━━━━━━━━━━━━━━━━━ 482s 2s/step - accuracy: 0.9655 - loss: 0.0963 - val_accuracy: 0.9509 - val_loss: 0.1406
Epoch 2/10
222/222 ━━━━━━━━━━━━━━━━━━━━ 322s 1s/step - accuracy: 0.9809 - loss: 0.0504 - val_accuracy: 0.9584 - val_loss: 0.1153
Epoch 3/10
222/222 ━━━━━━━━━━━━━━━━━━━━ 322s 1s/step - accuracy: 0.9824 - loss: 0.0427 - val_accuracy: 0.9575 - val_loss: 0.1091
Epoch 4/10
222/222 ━━━━━━━━━━━━━━━━━━━━ 386s 2s/step - accuracy: 0.9856 - loss: 0.0372 - val_accuracy: 0.9459 - val_loss: 0.1371
Epoch 5/10
222/222 ━━━━━━━━━━━━━━━━━━━━ 395s 2s/step - accuracy: 0.9881 - loss: 0.0326 - val_accuracy: 0.9575 - val_loss: 0.1051
Epoch 6/10
222/222 ━━━━━━━━━━━━━━━━━━━━ 319s 1s/step - accuracy: 0.9891 - loss: 0.0281 - val_accuracy: 0.9684 - val_loss: 0.1016
Epoch 7/10
222/222 ━━━━━━━━━━━━━━━━━━━━ 318s 1s/step - accuracy: 0.9891 - loss: 0.0304 - val_accuracy: 0.9559 - val_loss: 0.1095
Epoch 8/10
222/222 ━━━━━━━━━━━━━━━━━━━━ 318s 1s/step - accuracy: 0.9900 - loss: 0.0245 - val_accu

In [12]:
# đánh giá mô hình tren tập dữ liệu validation
val_loss, val_accuracy = model.evaluate(val_ds)
print(f'Validation Loss: {val_loss}')
print(f'Validation Accuracy: {val_accuracy}')

19/19 ━━━━━━━━━━━━━━━━━━━━ 21s 1s/step - accuracy: 0.9684 - loss: 0.0980
Validation Loss: 0.09801311790943146
Validation Accuracy: 0.9683597087860107


In [13]:
# Lưu mô hình
model.save('mobilenet_model.h5')